# 🔎 cryoDRGN — analyse a run **while it trains**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ts387/cryodrgn/blob/claude/cryodrgn-colab-notebook-5mf3p3/cryoDRGN_colab_live_analyze.ipynb)

Watch a `train_vae` run that is still going: plot its loss curves, and run
`cryodrgn analyze` on intermediate epochs as they land — without touching the training
job or waiting for it to finish.

| Step | What happens |
|------|--------------|
| 1. Setup | Install cryoDRGN, mount Drive |
| 2. Attach | Point at the running model folder; report progress and whether it is live |
| 3. Curves | Loss / KLD from `run.log` if it is visible — **latent drift** from `z.N.pkl` if it is not |
| 4. Analyze | `cryodrgn analyze` on the latest **complete** epoch |
| 5. View | The plots, inline |
| 6. Watch | Poll for new epochs and analyze each as it appears |

### ⚠️ `run.log` is normally invisible while training runs

`train_vae` attaches a `logging.FileHandler` to `<outdir>/run.log` (`train_vae.py:632`) and
holds it open for the entire run — it is only closed when the process exits. Google Drive's
FUSE mount uploads a file **when it is closed**, so from a *second* session `run.log` usually
does not exist at all until training finishes. `z.N.pkl` and `weights.N.pkl` are written and
closed every epoch, so those do sync, and everything here keys off them instead:

* **liveness** (2.2, 6.1) comes from the newest checkpoint's mtime, not `run.log`'s;
* **curves** (3.1) fall back to latent drift computed from consecutive `z.N.pkl`;
* **`cryodrgn analyze`** is pointed at a small local mirror of the run folder that carries a
  `run.log`, because `analysis.parse_loss` opens it unguarded (`analysis.py:31`) and would
  otherwise crash — *after* the slow UMAP step.

### Run this in a *second* Colab session

Colab executes one cell at a time per runtime, so the training cell blocks its own
notebook. Open this notebook separately — it reaches the run through Drive.

**A CPU runtime is enough** when `skip_volumes` is on: `cryodrgn analyze` never reads the
particle stack (only `config.yaml`, `z.N.pkl`, `weights.N.pkl` and `ctf.pkl`), and with
volume generation off there is nothing for a GPU to do. At ~0.26 units/hour that is
1/20th of an A100, and it leaves the training GPU alone.

### It does not disturb the training job

Everything it reads is read-only, and analysis output goes to a **separate folder** by
default rather than into the run directory. The only real hazard — reading a checkpoint
that is still being written — is handled in cell 2.2, which verifies an epoch is complete
before offering it.

## 1 · Setup

In [ ]:
#@title 1.1 · Install cryoDRGN { display-mode: "form" }
#@markdown Installs cryoDRGN from PyPI. Colab's pre-installed PyTorch/CUDA are kept.
#@markdown <br>• **stable** – the recommended release &nbsp;•&nbsp; **beta** – newest dev build from TestPyPI
release_channel = "stable"  #@param ["stable", "beta"]
#@markdown Optionally pin an exact version (e.g. `4.3.0`); leave blank for the latest.
version = ""  #@param {type:"string"}
#@markdown A few dependencies are pinned to versions other than Colab's defaults, so the
#@markdown runtime **restarts automatically** at the end. That is expected — just carry
#@markdown on with the next cell afterwards.
restart_after_install = True  #@param {type:"boolean"}

import subprocess, sys

pkg = "cryodrgn"
if version.strip():
    pkg = f"cryodrgn=={version.strip()}"

if release_channel == "beta":
    cmd = [sys.executable, "-m", "pip", "install", "-q",
           "-i", "https://test.pypi.org/simple/",
           "--extra-index-url", "https://pypi.org/simple/",
           "cryodrgn", "--pre"]
    if version.strip():
        cmd[cmd.index("cryodrgn")] = pkg
else:
    cmd = [sys.executable, "-m", "pip", "install", "-q", pkg]

print("Installing", pkg, f"({release_channel} channel) — this takes ~1-2 min...\n")
ret = subprocess.run(cmd)
if ret.returncode != 0:
    raise SystemExit("❌ pip install failed — see the log above.")

# --- realign torchvision with torch -------------------------------------------------
# cryoDRGN pins torch<2.10, so pip may DOWNGRADE Colab's torch. Colab's pre-installed
# torchvision was compiled against the newer torch, and once they disagree importing it
# raises "operator torchvision::nms does not exist". That breaks EVERY cryodrgn command,
# because the CLI eagerly imports all command modules and analyze_landscape_full imports
# umap -> torchvision. Matching pair is torch 2.N <-> torchvision 0.(N+15).
import importlib.metadata as md

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

tver, tvver = _ver("torch"), _ver("torchvision")
if tver and tvver:
    tmaj, tmin = (int(x) for x in tver.split(".")[:2])
    tvmin = int(tvver.split(".")[1])
    want = tmin + 15 if tmaj == 2 else None
    if want is not None and tvmin != want:
        print(f"\n⚠️  torch {tver} and torchvision {tvver} are incompatible "
              f"(cryoDRGN's torch<2.10 pin downgraded torch).")
        print(f"   Installing torchvision 0.{want}.* to match...")
        fix = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-deps",
             f"torchvision==0.{want}.*"])
        if fix.returncode == 0:
            print(f"   ✅ torchvision realigned to 0.{want}.*")
        else:
            print(f"   ❌ Could not install torchvision 0.{want}.* — if cryodrgn commands "
                  f"fail with 'torchvision::nms does not exist', run:")
            print(f"      !pip install --no-deps 'torchvision==0.{want}.*'")

print("\n✅ cryoDRGN installed.")
if restart_after_install:
    print("🔄 Restarting the runtime to finalize the install (this is normal)...")
    print("   When it reconnects, continue from cell 1.3 — do NOT re-run this cell.")
    get_ipython().kernel.do_shutdown(True)

In [ ]:
#@title 1.2 · Verify the installation { display-mode: "form" }
#@markdown Run this **after** the runtime has restarted. The `cryodrgn --version` smoke-test is
#@markdown the important one: the CLI imports *every* command module on startup, so a broken
#@markdown dependency anywhere makes all commands fail — better to catch it here than mid-run.
import sys, subprocess
import torch, cryodrgn

print(f"cryoDRGN version : {cryodrgn.__version__}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  CUDA not available — fine for Step 4 (CPU-only), but Step 6 needs a GPU (cell 1.1).")

# torchvision must match torch or `import umap` blows up inside the cryodrgn CLI
try:
    import torchvision
    print(f"torchvision      : {torchvision.__version__} (ok)")
except Exception as e:
    msg = str(e).splitlines()[0]
    want = "0.%d.*" % (int(torch.__version__.split(".")[1]) + 15)
    if "numpy.dtype size changed" in msg or "binary incompatibility" in msg:
        # cryoDRGN pins numpy<1.27, downgrading Colab's numpy 2.x. C extensions that were
        # compiled against numpy 2.x headers then fail their ABI check on import.
        print(f"⚠️  torchvision fails a NumPy ABI check: {msg}")
        print("   Cause: cryoDRGN pins numpy<1.27, so Colab's numpy 2.x was downgraded and")
        print("   torchvision (built against numpy 2.x) no longer matches.")
        print("   This is USUALLY HARMLESS: cryoDRGN never imports torchvision itself — only")
        print("   umap does, and cryodrgn.analysis imports umap lazily. Every cryodrgn command")
        print("   also runs as a subprocess. Treat the CLI check below as the real verdict, and")
        print("   don't 'fix' this unless something actually fails.")
    else:
        print(f"❌ torchvision is broken: {msg}")
        print("   Looks like a torch/torchvision version mismatch rather than a NumPy issue.")
        print(f"   Fix with:  !pip install --no-deps 'torchvision=={want}'")
        print("   then re-run this cell (no restart needed).")

print("\n$ cryodrgn --version")
r = subprocess.run(["cryodrgn", "--version"], capture_output=True, text=True)
print((r.stdout + r.stderr).strip()[-2000:])
if r.returncode != 0:
    raise RuntimeError("The cryodrgn CLI failed to start — fix the error above before continuing.")
print("\n✅ CLI healthy — all command modules import cleanly.")

In [ ]:
#@title 1.3 · Mount Google Drive { display-mode: "form" }
#@markdown Click the link that appears, pick your Google account, and paste the code
#@markdown (or approve the pop-up). Your Drive appears under `/content/drive/MyDrive`.
from google.colab import drive
drive.mount("/content/drive")
print("\n✅ Drive mounted at /content/drive/MyDrive")

## 2 · Attach to the running job

`train_vae` writes `weights.N.pkl` first and `z.N.pkl` second (`save_checkpoint`,
`train_vae.py:562-576`), so a readable `z.N.pkl` means epoch N is fully checkpointed. Cell
2.2 does not take that on trust: it loads the newest one and checks its shape against the
previous epoch's, stepping back if the file is still in flight. Over Drive's FUSE mount a
file can appear before its contents have flushed, so this matters.

In [ ]:
#@title 2.1 · Point at the run { display-mode: "form" }
#@markdown Drive project folder — the same one the training notebook uses.
drive_project_dir = "/content/drive/MyDrive/cryodrgn_project"  #@param {type:"string"}
#@markdown Model folder name inside it, e.g. `01_cryodrgn128_clean405k`.
run_name = ""  #@param {type:"string"}
#@markdown Where to put the analyses. Blank = `<project>/live_analysis/<run>` — deliberately
#@markdown outside the training folder so nothing can collide with the running job.
output_root = ""  #@param {type:"string"}

import os
if not run_name.strip():
    raise ValueError("Set run_name to the model folder you want to watch.")
DRIVE_DIR = os.path.abspath(drive_project_dir)
RUN = os.path.join(DRIVE_DIR, run_name.strip())
if not os.path.isdir(RUN):
    raise FileNotFoundError(f"{RUN} not found — check drive_project_dir / run_name.")
if not os.path.exists(os.path.join(RUN, "config.yaml")):
    raise FileNotFoundError(f"No config.yaml in {RUN} — has training written its first epoch?")
OUT = os.path.abspath(output_root.strip()) if output_root.strip() \
      else os.path.join(DRIVE_DIR, "live_analysis", run_name.strip())
os.makedirs(OUT, exist_ok=True)

for k, v in dict(LA_DRIVE=DRIVE_DIR, LA_RUN=RUN, LA_OUT=OUT).items():
    os.environ[k] = v

print(f"run        : {RUN}")
print(f"analyses → : {OUT}")
print("             (outside the run folder — the training job never sees these)")

In [ ]:
#@title 2.2 · Progress, and the latest *complete* epoch { display-mode: "form" }
#@markdown How long the newest checkpoint may go untouched before the job is called
#@markdown finished/stalled. Set this comfortably above one epoch of wall-clock time.
live_window_min = 45  #@param {type:"number"}

import os, re, glob, time
import numpy as np
import yaml
from cryodrgn import utils

RUN = os.environ["LA_RUN"]
cfg = yaml.safe_load(open(os.path.join(RUN, "config.yaml")))
zdim = cfg["model_args"]["zdim"]

def epochs_present():
    out = []
    for p in glob.glob(os.path.join(RUN, "z.*.pkl")):
        m = re.search(r"z\.(\d+)\.pkl$", os.path.basename(p))
        if m and int(m.group(1)) > 0:      # z.0.pkl is only ever a convergence-tool artifact
            out.append(int(m.group(1)))
    return sorted(out)

def load_ok(ep):
    """True if z.<ep>.pkl is fully written: unpickles, 2-D, right zdim, non-empty."""
    try:
        z = np.asarray(utils.load_pkl(os.path.join(RUN, f"z.{ep}.pkl")))
    except Exception:
        return None
    if z.ndim != 2 or z.shape[1] != zdim or z.shape[0] == 0:
        return None
    return z.shape[0]

eps = epochs_present()
if not eps:
    raise FileNotFoundError(f"No z.N.pkl in {RUN} yet — the first epoch has not finished.")

# walk back from the newest until one loads cleanly and matches its predecessor's row count
complete, nptcl = None, None
for ep in reversed(eps):
    n = load_ok(ep)
    if n is None:
        print(f"  epoch {ep}: z.{ep}.pkl not readable yet — still being written, skipping")
        continue
    prev = [e for e in eps if e < ep]
    if prev:
        n_prev = load_ok(prev[-1])
        if n_prev is not None and n_prev != n:
            print(f"  epoch {ep}: {n:,} rows vs {n_prev:,} at epoch {prev[-1]} — partial, skipping")
            continue
    complete, nptcl = ep, n
    break
if complete is None:
    raise RuntimeError("No completely-written epoch found. Wait a moment and re-run.")

# --- liveness comes from the CHECKPOINTS, not run.log --------------------------------
# train_vae holds run.log open for the whole run (train_vae.py:632) and Drive's FUSE mount
# only uploads a file when it is closed, so from this session run.log usually does not exist
# at all until training exits.  z.N.pkl / weights.N.pkl are closed every epoch, so their
# mtimes are the signal that actually tracks the job.
ckpts = glob.glob(os.path.join(RUN, "z.*.pkl")) + glob.glob(os.path.join(RUN, "weights.*.pkl"))
newest = max((os.path.getmtime(p) for p in ckpts), default=None)
age_min = (time.time() - newest) / 60 if newest else float("inf")
live = age_min < float(live_window_min)

log = os.path.join(RUN, "run.log")
has_log = os.path.exists(log)

# how far is it meant to go? the last Namespace line in run.log records num_epochs — only
# available once run.log lands, i.e. usually only after the run has finished.
target = None
if has_log:
    for line in open(log):
        if "Namespace(" in line:
            m = re.search(r"num_epochs=(\d+)", line)
            if m:
                target = int(m.group(1))

print(f"particles      : {nptcl:,}   zdim {zdim}")
print(f"epochs written : {len(eps)}  (up to {max(eps)})")
print(f"latest COMPLETE: {complete}" + (f" of {target}" if target else ""))
print(f"newest ckpt    : written {age_min:.1f} min ago -> "
      f"{'TRAINING LIVE' if live else 'idle (finished, stalled, or the session died)'}")
if target and complete >= target:
    print("               : target reached — the job should be running its final analyze")

print("\nrun folder contents")
for name, note in [("config.yaml", "required by analyze"),
                   ("run.log", "expected MISSING while training runs — see below"),
                   ("z.pkl", "final latent, written only at the end"),
                   ("weights.pkl", "final weights, written only at the end")]:
    p = os.path.join(RUN, name)
    print(f"  {name:<14} {'present' if os.path.exists(p) else 'absent ':<8}  ({note})")
print(f"  z.N.pkl        {len(eps):<3} file(s)   weights.N.pkl "
      f"{len(glob.glob(os.path.join(RUN, 'weights.*.pkl')))} file(s)")

if has_log:
    tail = [l.rstrip() for l in open(log) if "=====>" in l][-3:]
    if tail:
        print("\nlast epoch lines from run.log:")
        for l in tail:
            print("   " + l[-140:])
else:
    print("\nℹ️  run.log is not visible from this session — that is the expected state for a")
    print("   running job, not a fault. train_vae keeps it open and Drive only uploads a file")
    print("   on close, so it appears once training exits. Cell 3.1 plots latent drift from")
    print("   the z.N.pkl files instead, and cell 4.1 supplies analyze with its own run.log.")

os.environ["LA_COMPLETE"] = str(complete)
os.environ["LA_ZDIM"] = str(zdim)
os.environ["LA_HAS_LOG"] = "1" if has_log else "0"

## 3 · Curves

**If `run.log` is visible** (i.e. the run has finished, or you pointed the cell at a copy) you
get the loss / KLD curves parsed straight out of it.

**While training is still going it usually is not** — see the note at the top — so the cell
falls back to **latent drift**, computed from the `z.N.pkl` files, which *do* sync:

| Metric | What it is | Reading it |
|---|---|---|
| median ‖zₙ − zₙ₋₁‖ | how far a typical particle's embedding moved this epoch | falling → the latent is settling |
| median cosine distance | angle between one epoch's move and the next | → **1.0** = consecutive moves uncorrelated, i.e. jitter rather than progress; near 0 = still travelling in a consistent direction |
| median ‖zₙ‖ | how spread out the latent is | should plateau |

These are the same two quantities `cryodrgn analyze_convergence` plots as
`encoder_latent_vector_shifts` (`analyze_convergence.py:342-397`), computed here on the fly so
you do not have to wait for the run to end. They are the better convergence signal anyway —
the total loss keeps creeping down long after the latent has stopped moving.

Each epoch is one `N × zdim × 4` byte read from Drive (13 MB at 405k particles, zdim 8), cached
in memory so re-running the cell only reads what is new.

In [ ]:
#@title 3.1 · Loss curves, or latent drift if run.log is not visible { display-mode: "form" }
#@markdown Optional path to a readable copy of `run.log`. Blank = the one in the run folder
#@markdown (normally absent until training exits).
runlog_path = ""  #@param {type:"string"}
#@markdown Plot latent drift as well, even when the loss curves are available.
always_show_drift = True  #@param {type:"boolean"}
#@markdown Use only the newest N epochs for the drift curve (`0` = all).
drift_last_n = 0  #@param {type:"integer"}

import os, re, glob
import numpy as np
import matplotlib.pyplot as plt
from cryodrgn import utils

RUN = os.environ["LA_RUN"]
log = os.path.abspath(runlog_path.strip()) if runlog_path.strip() \
      else os.path.join(RUN, "run.log")

# ---------- (a) loss curves, if run.log happens to be readable -----------------------
pat = re.compile(r"Epoch:\s*(\d+)\s+Average gen loss\s*=\s*([\d.eE+-]+),\s*"
                 r"KLD\s*=\s*([\d.eE+-]+),\s*total loss\s*=\s*([\d.eE+-]+)")
rows = {}                       # dict: a resumed run re-appends, last write per epoch wins
if os.path.exists(log):
    with open(log) as f:
        for line in f:
            m = pat.search(line)
            if m:
                rows[int(m.group(1))] = tuple(float(m.group(i)) for i in (2, 3, 4))
    if not rows:
        print(f"{log} exists but has no epoch-summary lines yet.")
else:
    print(f"{log} not found — expected while training runs: train_vae holds run.log open")
    print("(train_vae.py:632) and Drive only uploads a file on close. Plotting latent drift")
    print("from the z.N.pkl files instead, which are closed every epoch and so do sync.\n")

if rows:
    ep = np.array(sorted(rows))
    gen, kld, tot = (np.array([rows[e][i] for e in ep]) for i in range(3))
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
    for ax, y, nm in zip(axes, (tot, gen, kld),
                         ("total loss", "reconstruction (gen)", "KLD")):
        ax.plot(ep, y, marker="o", ms=3)
        ax.set_xlabel("epoch"); ax.set_ylabel(nm); ax.grid(alpha=0.3)
    fig.suptitle("training loss (from run.log)", y=1.04)
    fig.tight_layout(); plt.show()

    print(f"{len(ep)} epochs parsed (epochs {ep.min()}-{ep.max()})")
    if len(ep) >= 6:
        d = (tot[-1] - tot[-6]) / abs(tot[-6]) * 100
        print(f"total loss over the last 5 epochs: {d:+.3f}%  "
              f"({'still improving' if d < -0.05 else 'flattening out'})")
    print("KLD:", f"{kld[0]:.3f} -> {kld[-1]:.3f}",
          "(rising)" if kld[-1] > kld[0] else "(falling)")

# ---------- (b) latent drift, straight from the checkpoints --------------------------
if not rows or always_show_drift:
    zcache = globals().setdefault("_LA_Z_CACHE", {})     # survives re-runs of this cell

    def _z(e):
        if e not in zcache:
            zcache[e] = np.asarray(utils.load_pkl(os.path.join(RUN, f"z.{e}.pkl")),
                                   dtype=np.float32)
        return zcache[e]

    deps = sorted(int(m.group(1))
                  for p in glob.glob(os.path.join(RUN, "z.*.pkl"))
                  for m in [re.search(r"z\.(\d+)\.pkl$", os.path.basename(p))]
                  if m and int(m.group(1)) > 0)
    if int(drift_last_n) > 0:
        deps = deps[-int(drift_last_n):]
    if len(deps) < 2:
        print("Need at least 2 epochs for a drift curve — only "
              f"{len(deps)} z.N.pkl found so far.")
    else:
        print(f"reading {len(deps)} latent files (epochs {deps[0]}-{deps[-1]})"
              f"{' — cached from a previous run of this cell' if len(zcache) >= len(deps) else ''}")
        mag, mag_x, cos, cos_x, spread, spread_x = [], [], [], [], [], []
        prev_d, prev_e = None, None
        for e in deps:
            try:
                ze = _z(e)
            except Exception as exc:            # mid-write on the FUSE mount
                print(f"  epoch {e}: not readable yet ({type(exc).__name__}) — skipping")
                prev_d, prev_e = None, None
                continue
            spread.append(float(np.median(np.linalg.norm(ze, axis=1)))); spread_x.append(e)
            if prev_e is not None and _z(prev_e).shape == ze.shape:
                d = ze - _z(prev_e)
                mag.append(float(np.median(np.linalg.norm(d, axis=1)))); mag_x.append(e)
                if prev_d is not None:
                    uv = np.einsum("ij,ij->i", d, prev_d)
                    nu, nv = np.linalg.norm(d, axis=1), np.linalg.norm(prev_d, axis=1)
                    ok = (nu > 0) & (nv > 0)
                    if ok.any():
                        cos.append(float(np.median(1 - uv[ok] / (nu[ok] * nv[ok]))))
                        cos_x.append(e)
                prev_d = d
            else:
                prev_d = None
            prev_e = e

        fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
        for ax, x, y, nm in [
            (axes[0], mag_x, mag, "median |z$_n$ - z$_{n-1}$|"),
            (axes[1], cos_x, cos, "median cosine distance"),
            (axes[2], spread_x, spread, "median |z|"),
        ]:
            if y and len(x) == len(y):
                ax.plot(x, y, marker="o", ms=3)
            ax.set_xlabel("epoch"); ax.set_ylabel(nm); ax.grid(alpha=0.3)
        axes[1].axhline(1.0, ls="--", c="grey", lw=1)     # 1 = orthogonal
        axes[1].set_ylim(0, 2)                            # 1 - cos(theta) spans [0, 2]
        fig.suptitle("latent drift (from z.N.pkl)", y=1.04)
        fig.tight_layout(); plt.show()

        if len(mag) >= 2:
            print(f"per-epoch latent movement: {mag[0]:.4f} -> {mag[-1]:.4f}  "
                  f"({(mag[-1]-mag[0])/mag[0]*100:+.1f}%)  "
                  f"{'settling' if mag[-1] < mag[0] else 'not settling yet'}")
        if cos:
            c = cos[-1]
            print(f"cosine distance now {c:.3f}  " + (
                "(near 0 = each epoch carries on in the last one's direction — still "
                "travelling, keep training)" if c < 0.6 else
                "(near 1 = consecutive moves are uncorrelated, i.e. jitter rather than "
                "progress — the latent has converged)" if c <= 1.2 else
                "(above 1 = consecutive moves oppose each other — oscillating about a "
                "settled point, also a converged signature)"))
#@markdown ---
#@markdown Whatever is plotted here is measured on **training** data — `train_vae` keeps no
#@markdown held-out set, so neither a falling loss nor a settling latent says anything about
#@markdown generalisation.

## 4 · Analyze an intermediate epoch

`cryodrgn analyze` reads only `config.yaml`, `z.N.pkl`, `run.log`, and — for volumes —
`weights.N.pkl`, plus `ctf.pkl` for the pixel size. It never opens the particle stack, so this
is cheap and cannot contend with the trainer for data.

**It needs a `run.log` to exist.** `analysis.parse_loss` does a bare `open(f"{workdir}/run.log")`
(`analysis.py:31`), called from `analyze_zN` at the plotting step (`analyze.py:184`) — *after*
UMAP. On a live run that file is not visible here, so analyze would grind through the slow part
and then die with `FileNotFoundError`. The cell therefore runs analyze against a **local mirror**
of the run folder under `/content/la_workdir/<run>/`, holding just `config.yaml`, the epoch's
checkpoint, and a `run.log` (the real one if it has landed, otherwise an empty file, which
`parse_loss` handles fine — the learning-curve PNG just comes out blank). The Drive run folder
is never written to, and repeat analyses of the same epoch skip the download.

`skip_volumes` decides whether you need a GPU. With it on you get the PCA, UMAP and k-means
labels — everything that tells you how the latent space is developing — on CPU alone, and
`weights.N.pkl` (a few hundred MB) is never fetched, since `VolumeGenerator.gen_volumes`
returns immediately (`analyze.py:436-438`). Turn it off only when you want the k-means volumes.

In [ ]:
#@title 4.1 · Run analyze on one epoch { display-mode: "form" }
#@markdown Epoch to analyze. **-1** uses the latest complete one found in 2.2.
epoch = -1  #@param {type:"integer"}
#@markdown Skip volume generation — keeps it CPU-only and quick.
skip_volumes = True  #@param {type:"boolean"}
#@markdown Skip UMAP as well (PCA only). UMAP is the slow part on CPU: minutes at 400k particles.
skip_umap = False  #@param {type:"boolean"}
#@markdown k-means samples (ignored when skip_volumes is on for volume purposes, still labels z).
ksample = 20  #@param {type:"integer"}
#@markdown Pixel size — `0` resolves it from `ctf.pkl` as `cryodrgn analyze` itself would.
apix = 0  #@param {type:"number"}

import os, time, shutil
import numpy as np
import yaml
from cryodrgn import utils

RUN, OUT = os.environ["LA_RUN"], os.environ["LA_OUT"]
ep = int(epoch) if int(epoch) >= 0 else int(os.environ["LA_COMPLETE"])
if not os.path.exists(os.path.join(RUN, f"z.{ep}.pkl")):
    raise FileNotFoundError(f"No z.{ep}.pkl in {RUN}.")

adir = os.path.join(OUT, f"analyze.{ep}")
cfg = yaml.safe_load(open(os.path.join(RUN, "config.yaml")))

# --- local mirror of the run folder --------------------------------------------------
# analyze does an unguarded open() on <workdir>/run.log (analysis.py:31) at the plotting
# step, i.e. AFTER UMAP.  While training runs that file is not visible on Drive, so point
# analyze at a small local copy that has one.  Nothing is written back to RUN.
SHADOW = os.path.join("/content/la_workdir", os.path.basename(RUN.rstrip("/")))
os.makedirs(SHADOW, exist_ok=True)
shutil.copyfile(os.path.join(RUN, "config.yaml"), os.path.join(SHADOW, "config.yaml"))
_srclog, _dstlog = os.path.join(RUN, "run.log"), os.path.join(SHADOW, "run.log")
if os.path.exists(_srclog):
    shutil.copyfile(_srclog, _dstlog)          # finished run: real learning curve
elif not os.path.exists(_dstlog):
    open(_dstlog, "w").close()                 # live run: empty -> blank learning curve
    print("run.log   : not on Drive yet (training holds it open) — analyze gets an empty one,")
    print("            so learning_curve_epoch*.png will be blank. Use cell 3.1 for curves.")
# weights.N.pkl is only read when volumes are generated, so don't fetch it otherwise
for _f in [f"z.{ep}.pkl"] + ([] if skip_volumes else [f"weights.{ep}.pkl"]):
    _s, _d = os.path.join(RUN, _f), os.path.join(SHADOW, _f)
    if not os.path.exists(_d) or os.path.getsize(_d) != os.path.getsize(_s):
        print(f"staging   : {_f} ({os.path.getsize(_s)/2**20:.0f} MB)", flush=True)
        shutil.copyfile(_s, _d)

# analyze infers A/px from ctf.pkl and rescales for the box (analyze.py:462-488); it does
# that itself when --Apix is omitted, so only pass one when the user overrides.
use_apix = float(apix)
if use_apix <= 0:
    _ctf = cfg["dataset_args"].get("ctf")
    if _ctf and os.path.exists(_ctf):
        cp = np.asarray(utils.load_pkl(_ctf))
        ap = set(cp[:, 1])
        if len(ap) == 1:
            cur = round(tuple(ap)[0] * tuple(set(cp[:, 0]))[0]
                        / (cfg["lattice_args"]["D"] - 1), 6)
            print(f"A/px      : {cur} (analyze will infer this from {os.path.basename(_ctf)})")
    else:
        print("A/px      : ctf.pkl not reachable — analyze will fall back to 1.0")

cmd = f'cryodrgn analyze "{SHADOW}" {ep} -o "{adir}" --ksample {int(ksample)}'
if skip_volumes:
    cmd += " --skip-vol"
if skip_umap:
    cmd += " --skip-umap"
if use_apix > 0:
    cmd += f" --Apix {use_apix}"

print(f"epoch     : {ep}")
print(f"workdir   : {SHADOW}   (local mirror; the Drive run folder is untouched)")
print(f"output    : {adir}   (NOT inside the run folder)")
print("$", cmd, "\n" + "=" * 70)
t0 = time.time()
get_ipython().system(cmd)
_rc = get_ipython().user_ns.get("_exit_code", 0)
if _rc:
    raise RuntimeError(f"Command failed (exit {_rc}) — see the log above.")
print("=" * 70 + f"\n✅ epoch {ep} analysed in {(time.time()-t0)/60:.1f} min → {adir}")
os.environ["LA_LAST"] = str(ep)

## 5 · View

In [ ]:
#@title 5.1 · View the plots { display-mode: "form" }
#@markdown Epoch to display — **-1** shows the most recently analysed.
epoch = -1  #@param {type:"integer"}

import os, re, glob
from IPython.display import Image, display, Markdown

OUT = os.environ["LA_OUT"]
if int(epoch) >= 0:
    ep = int(epoch)
else:
    found = [int(m.group(1)) for p in glob.glob(os.path.join(OUT, "analyze.*"))
             for m in [re.search(r"analyze\.(\d+)$", p)] if m]
    if not found:
        raise FileNotFoundError(f"Nothing analysed yet under {OUT} — run 4.1.")
    ep = max(found)
adir = os.path.join(OUT, f"analyze.{ep}")

display(Markdown(f"### epoch {ep}"))
shown = 0
for fname, cap in [
    ("z_pca.png", "**PCA** of the latent embeddings, coloured by k-means cluster"),
    ("umap.png", "**UMAP** of the latent embeddings"),
    ("z_pca_marginals.png", "PCA with marginal distributions"),
    ("umap_marginals.png", "UMAP with marginal distributions"),
    (f"learning_curve_epoch{ep}.png",
     "Training loss to this epoch — **blank on a live run**, because analyze builds it by "
     "parsing `run.log`, which is not visible until training exits. Cell 3.1 is the "
     "substitute."),
]:
    hits = glob.glob(os.path.join(adir, fname)) or glob.glob(os.path.join(adir, "kmeans*", fname))
    if hits:
        display(Markdown(cap)); display(Image(hits[0], width=520)); shown += 1
print(f"{shown} plot(s) from {adir}")

others = sorted(int(m.group(1)) for p in glob.glob(os.path.join(OUT, "analyze.*"))
                for m in [re.search(r"analyze\.(\d+)$", p)] if m)
if len(others) > 1:
    print(f"also analysed: {others}  — set epoch above to compare how the latent developed")

## 6 · Watch mode

Polls for new completed epochs and analyses each as it appears. Stop the cell whenever you
like — nothing is left half-done, and re-running picks up where it left off since epochs
already analysed are skipped.

The poll interval should be a decent fraction of your epoch time: checking every 30 s when an
epoch takes 15 minutes just burns Drive requests.

In [ ]:
#@title 6.1 · Watch for new epochs and analyze them { display-mode: "form" }
#@markdown Seconds between checks.
poll_seconds = 300  #@param {type:"integer"}
#@markdown Stop after this many analyses (`0` = keep going until you interrupt).
max_analyses = 0  #@param {type:"integer"}
#@markdown Analyze only every Nth epoch (`1` = every one).
every_n_epochs = 5  #@param {type:"integer"}
#@markdown Same options as 4.1.
skip_volumes = True  #@param {type:"boolean"}
skip_umap = False  #@param {type:"boolean"}
#@markdown Give up if no checkpoint has been written for this long — training has stopped.
#@markdown (Checkpoint mtimes, not `run.log`: that file is invisible here until the run ends.)
idle_stop_min = 45  #@param {type:"number"}

import os, re, glob, time, shutil
import numpy as np
import yaml
from cryodrgn import utils
from IPython.display import clear_output

RUN, OUT = os.environ["LA_RUN"], os.environ["LA_OUT"]
zdim = int(os.environ["LA_ZDIM"])

# same local mirror as 4.1 — analyze needs a run.log to exist (analysis.py:31)
SHADOW = os.path.join("/content/la_workdir", os.path.basename(RUN.rstrip("/")))
os.makedirs(SHADOW, exist_ok=True)

def stage(ep):
    """Mirror what analyze needs for epoch ep into SHADOW; return SHADOW."""
    shutil.copyfile(os.path.join(RUN, "config.yaml"), os.path.join(SHADOW, "config.yaml"))
    s_log, d_log = os.path.join(RUN, "run.log"), os.path.join(SHADOW, "run.log")
    if os.path.exists(s_log):
        shutil.copyfile(s_log, d_log)
    elif not os.path.exists(d_log):
        open(d_log, "w").close()
    for f in [f"z.{ep}.pkl"] + ([] if skip_volumes else [f"weights.{ep}.pkl"]):
        s, d = os.path.join(RUN, f), os.path.join(SHADOW, f)
        if not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s):
            shutil.copyfile(s, d)
    return SHADOW

def ckpt_age_min():
    """Minutes since the newest checkpoint was written — the real liveness signal."""
    ck = (glob.glob(os.path.join(RUN, "z.*.pkl"))
          + glob.glob(os.path.join(RUN, "weights.*.pkl")))
    if not ck:
        return float("inf")
    return (time.time() - max(os.path.getmtime(p) for p in ck)) / 60

def complete_epochs():
    """Epochs whose z.N.pkl is fully written."""
    out = []
    for p in sorted(glob.glob(os.path.join(RUN, "z.*.pkl"))):
        m = re.search(r"z\.(\d+)\.pkl$", os.path.basename(p))
        if not m or int(m.group(1)) == 0:
            continue
        try:
            z = np.asarray(utils.load_pkl(p))
        except Exception:
            continue                      # mid-write; it will be picked up next poll
        if z.ndim == 2 and z.shape[1] == zdim and z.shape[0] > 0:
            out.append(int(m.group(1)))
    return sorted(out)

def already_done():
    return {int(m.group(1)) for p in glob.glob(os.path.join(OUT, "analyze.*"))
            for m in [re.search(r"analyze\.(\d+)$", p)] if m
            and os.path.exists(os.path.join(p, "z_pca.png"))}

n_done, t_start = 0, time.time()
print(f"watching {RUN}\nevery {poll_seconds}s, analysing every {every_n_epochs} epoch(s)\n"
      f"stop the cell to quit\n" + "=" * 70)
try:
    while True:
        done, avail = already_done(), complete_epochs()
        todo = [e for e in avail if e % int(every_n_epochs) == 0 and e not in done]
        age = ckpt_age_min()

        if todo:
            ep = todo[-1]                 # newest first — the stale ones matter less
            print(f"\n[{time.strftime('%H:%M:%S')}] epoch {ep} is ready — analysing")
            adir = os.path.join(OUT, f"analyze.{ep}")
            cmd = f'cryodrgn analyze "{stage(ep)}" {ep} -o "{adir}" --ksample 20'
            if skip_volumes:
                cmd += " --skip-vol"
            if skip_umap:
                cmd += " --skip-umap"
            print("   $", cmd, flush=True)
            t0 = time.time()
            get_ipython().system(cmd)
            rc = get_ipython().user_ns.get("_exit_code", 0)
            if rc:
                print(f"   ⚠️  analyze failed (exit {rc}) — will retry on the next poll")
            else:
                n_done += 1
                print(f"   ✅ {(time.time()-t0)/60:.1f} min → {adir}   "
                      f"({n_done} analysed this session)")
            if int(max_analyses) and n_done >= int(max_analyses):
                print(f"\nreached max_analyses={max_analyses} — stopping.")
                break
            continue                      # look again straight away in case more are queued

        if age > float(idle_stop_min):
            print(f"\nno new checkpoint for {age:.0f} min (> idle_stop_min) — training has "
                  f"stopped. Analysed {n_done} epoch(s) this session.")
            break

        print(f"[{time.strftime('%H:%M:%S')}] complete: {max(avail) if avail else '-'}   "
              f"analysed: {sorted(done) if done else '-'}   "
              f"newest ckpt {age:.0f} min old   next check in {poll_seconds}s", flush=True)
        time.sleep(int(poll_seconds))
except KeyboardInterrupt:
    print(f"\n⏹ stopped. {n_done} epoch(s) analysed this session; "
          f"training is unaffected and still running.")

---
### Notes

**Why a separate output folder.** The trainer owns its run directory and will write its own
`analyze.<final>/` when it finishes. Keeping these analyses in `<project>/live_analysis/<run>/`
means the two can never race, and the run folder stays exactly as `train_vae` left it — which
matters because cell 6.2 of the main notebook rebuilds a resume from `config.yaml` in there.

**What "complete" means.** `save_checkpoint` writes `weights.N.pkl` and then `z.N.pkl`, so a
readable `z.N.pkl` implies both are done. The checks here go further and confirm the array
unpickles, is 2-D, has the right `zdim` and matches the previous epoch's particle count —
because on a FUSE mount a file can be visible before its bytes are.

**Why `run.log` is missing, and what follows from it.** `train_vae` does
`logger.addHandler(logging.FileHandler(f"{args.outdir}/run.log"))` (`train_vae.py:632`). A
`FileHandler` opens with `mode='a'` and keeps the stream open until the process exits, and
Drive's FUSE mount only uploads a file when it is closed — so a second session sees nothing
there until training ends. Three consequences, all handled above:

1. Liveness cannot be read from `run.log`'s mtime; cells 2.2 and 6.1 use the newest
   `z.*.pkl` / `weights.*.pkl` mtime instead.
2. `cryodrgn analyze` would crash on it — `analysis.parse_loss` opens it unguarded
   (`analysis.py:31`), from `analyze_zN` at `analyze.py:184`, *after* UMAP has run. Cells 4.1
   and 6.1 give it a local mirror workdir with a `run.log` of its own. The
   `learning_curve_epoch*.png` it emits is consequently blank on a live run.
3. Loss and KLD numbers only exist in `run.log`, so cell 3.1 falls back to latent drift.
   If you want the real curves live, add a line to the *training* notebook that copies
   `run.log` to a second name every so often — a closed copy does sync — and point 3.1's
   `runlog_path` at it. Nothing here requires that.

Once training exits, `run.log` lands in the run folder and every cell picks it up automatically:
2.2 will report the target epoch count, 3.1 will plot the real loss curves, and analyze will
produce a populated learning curve. Delete the stale mirror at `/content/la_workdir/<run>/` (or
just re-run 4.1, which re-copies `run.log` whenever it is present) if you re-analyse afterwards.

**What you can actually tell from an intermediate epoch.** The UMAP and PCA show whether the
latent has settled into its final shape; comparing two epochs a few apart is the quickest read
on that. What none of it can show is overfitting — there is no held-out set (see the notes on
`eval_images` in the main notebook).

**Cost.** With `skip_volumes` on, a CPU high-RAM runtime at ~0.26 units/hour is plenty. UMAP
on several hundred thousand particles is the slow step, a few minutes per epoch; `skip_umap`
drops that to seconds if you only want the PCA and the loss curve.